In [1]:
# Importando as bibliotecas

import pandas as pd 
import numpy as np
import os
from datetime import datetime, date
from google.cloud import bigquery
from dotenv import load_dotenv
import pyarrow


# Carregando variáveis de ambiente
load_dotenv()

True

In [2]:
df_2023_01 = pd.read_csv("/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/Preços semestrais - AUTOMOTIVOS_2023.01.csv", sep=';')

In [3]:
df_2023_02 = pd.read_csv("/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/Preços semestrais - AUTOMOTIVOS_2023.02.csv", sep=';')

In [4]:
df_2023 = pd.concat((df_2023_01, df_2023_02))

In [5]:
df_2023.info()

<class 'pandas.core.frame.DataFrame'>
Index: 904000 entries, 0 to 472423
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Regiao - Sigla     904000 non-null  object 
 1   Estado - Sigla     904000 non-null  object 
 2   Municipio          904000 non-null  object 
 3   Revenda            904000 non-null  object 
 4   CNPJ da Revenda    904000 non-null  object 
 5   Nome da Rua        904000 non-null  object 
 6   Numero Rua         903753 non-null  object 
 7   Complemento        208421 non-null  object 
 8   Bairro             902318 non-null  object 
 9   Cep                904000 non-null  object 
 10  Produto            904000 non-null  object 
 11  Data da Coleta     904000 non-null  object 
 12  Valor de Venda     904000 non-null  object 
 13  Valor de Compra    0 non-null       float64
 14  Unidade de Medida  904000 non-null  object 
 15  Bandeira           904000 non-null  object 
dtypes: floa

In [6]:
df_2023_pe = df_2023[df_2023["Estado - Sigla"] == "PE"]

In [7]:
df_2023_pe.head()

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
1586,NE,PE,ARARIPINA,POSTO BH AUTO CENTRO LTDA,08.140.286/0001-12,RUA DIONISIO DE DEUS LIMA,57,NaN,CENTRO,56280-000,DIESEL S10,03/01/2023,"6,98",NaN,R$ / litro,VIBRA ENERGIA
1587,NE,PE,ARARIPINA,POSTO BH AUTO CENTRO LTDA,08.140.286/0001-12,RUA DIONISIO DE DEUS LIMA,57,NaN,CENTRO,56280-000,GASOLINA ADITIVADA,03/01/2023,"5,49",NaN,R$ / litro,VIBRA ENERGIA
1588,NE,PE,ARARIPINA,POSTO BH AUTO CENTRO LTDA,08.140.286/0001-12,RUA DIONISIO DE DEUS LIMA,57,NaN,CENTRO,56280-000,GASOLINA,03/01/2023,"5,49",NaN,R$ / litro,VIBRA ENERGIA
1589,NE,PE,ARARIPINA,JUVENAL ANGELO & CIA LTDA,04.965.564/0003-81,AVENIDA GOVERNADOR MUNIZ FALCAO,525,A,CENTRO,56280-000,ETANOL,03/01/2023,"4,25",NaN,R$ / litro,RAIZEN
1590,NE,PE,ARARIPINA,JUVENAL ANGELO & CIA LTDA,04.965.564/0003-81,AVENIDA GOVERNADOR MUNIZ FALCAO,525,A,CENTRO,56280-000,DIESEL S10,03/01/2023,"6,19",NaN,R$ / litro,RAIZEN


In [ ]:
# Configurando credenciais

# Caminho do arquivo de credenciais GCP
credencial = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Identificador do projeto no GCP
project_id = os.getenv("PROJECT_ID")

# Tabela Bronze no BigQuery
table_id = os.getenv("BRONZE_2023")

# Inicialização do cliente BigQuery

# Define a variavel de ambiente
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = credencial

# Cria o cliente Bigquery ao projeto configurado
client = bigquery.Client(project=project_id)

In [9]:
# Criação e carga da tabela BRONZE

# Configuração do job de carga no BigQuery
# WRITE_TRUNCATE garante que a tabela Bronze seja sempre sobrescrita,
# mantendo apenas o retrato mais recente dos dados de origem
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

# Envia o DataFrame para o BigQuery,
# criando a tabela automaticamente caso ainda não exista
job = client.load_table_from_dataframe(
    df_2023_pe,
    table_id,
    job_config=job_config
)

# Aguarda a finalização do job para garantir consistência da carga
job.result()

# Confirmação visual de sucesso da operação
print("✅ Tabela Bronze criada e dados carregados com sucesso")

/home/danielpedro/anaconda3/envs/combustivel_projeto/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✅ Tabela Bronze criada e dados carregados com sucesso
